# PyTorch Bi-LSTM Poem Generator

This notebook trains a compact PyTorch port of the Bi-Directional LSTM poem generator architecture inspired by `Reema-Khaseeb/NLP-poem-generator`.

Expected input data:

- Preferred: `model/data/poem.txt`
- Colab fallback: `poem.txt` in the current notebook directory

Saved artifacts:

- `model/data/weights/reema_bi_lstm_poem_generator.pt`
- `model/data/vocabulary.json`
- `model/data/training_config.json`
- `model/data/training_history.json`

Safety note: this notebook does not run shell commands, download remote files, mount drives, read secrets, or load pickle files from unknown sources. It only reads the local poem corpus and writes local training artifacts.


In [ ]:
from collections import Counter
import json
import math
from pathlib import Path
import random
import re

import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

DATA_CANDIDATES = [Path('model/data/poem.txt'), Path('poem.txt')]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Upload poem.txt or keep it at model/data/poem.txt before running the notebook.')

DATA_DIR = Path('model/data')
WEIGHTS_DIR = DATA_DIR / 'weights'
DATA_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS_PATH = WEIGHTS_DIR / 'reema_bi_lstm_poem_generator.pt'
VOCAB_PATH = DATA_DIR / 'vocabulary.json'
CONFIG_PATH = DATA_DIR / 'training_config.json'
HISTORY_PATH = DATA_DIR / 'training_history.json'
SOURCES_PATH = DATA_DIR / 'corpus_sources.json'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = device.type == 'cuda'
print(f'Using device: {device}')
print(f'Using data file: {DATA_PATH}')


## Load And Normalize The Corpus

The corpus is treated as plain text. The cleanup is intentionally conservative: normalize quotes/spacing, skip very short lines, and skip obvious all-caps headings so the model learns poem language rather than table-of-contents noise.


In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:['’-][A-Za-z]+)*")
ROMAN_RE = re.compile(r'[IVXLCDM]+')


def normalize_line(line: str) -> str:
    line = line.replace('’', "'").replace('`', "'")
    line = re.sub(r'\s+', ' ', line.strip())
    return line


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(normalize_line(text).lower())


def is_probably_heading(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return True
    if ROMAN_RE.fullmatch(stripped):
        return True
    words = stripped.split()
    has_letter = any(char.isalpha() for char in stripped)
    return has_letter and len(words) <= 8 and stripped.upper() == stripped


raw_lines = DATA_PATH.read_text(encoding='utf-8').splitlines()
corpus = []
tokenized_lines = []
for line in raw_lines:
    cleaned = normalize_line(line)
    tokens = tokenize(cleaned)
    if len(tokens) >= 2 and not is_probably_heading(cleaned):
        corpus.append(cleaned)
        tokenized_lines.append(tokens)

if len(tokenized_lines) < 10:
    raise ValueError('Not enough usable training lines were found in the poem corpus.')

print(f'Raw lines: {len(raw_lines)}')
print(f'Usable poem lines: {len(tokenized_lines)}')
print(f'Total usable tokens: {sum(len(tokens) for tokens in tokenized_lines)}')
print(corpus[:3])

if SOURCES_PATH.exists():
    sources = json.loads(SOURCES_PATH.read_text(encoding='utf-8'))
    print('Corpus source metadata found:', [source['name'] for source in sources.get('sources', [])])


## Line-Level Train/Validation Split

Instead of randomly splitting n-gram examples, we split whole poem lines first. This avoids leaking prefixes from the same line into both train and validation.


In [ ]:
VALIDATION_SPLIT = 0.12

split_generator = torch.Generator().manual_seed(SEED)
line_indices = torch.randperm(len(tokenized_lines), generator=split_generator).tolist()
val_line_count = max(1, int(len(line_indices) * VALIDATION_SPLIT))
val_line_indices = set(line_indices[:val_line_count])

train_tokenized_lines = [tokens for index, tokens in enumerate(tokenized_lines) if index not in val_line_indices]
val_tokenized_lines = [tokens for index, tokens in enumerate(tokenized_lines) if index in val_line_indices]

print(f'Train lines: {len(train_tokenized_lines)}')
print(f'Validation lines: {len(val_tokenized_lines)}')


## Build Vocabulary And Next-Word Examples

Rare words are useful poetically, but a tiny corpus plus a huge word-level softmax makes training unstable. `MIN_WORD_FREQ = 2` keeps the target space learnable while still preserving uncommon words in the context as `<unk>`.


In [ ]:
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN]
MIN_WORD_FREQ = 2
MAX_VOCAB_SIZE = 12000
MAX_SEQUENCE_LEN_CAP = 32

counter = Counter(token for line in train_tokenized_lines for token in line)
vocab_words = [word for word, count in counter.most_common() if count >= MIN_WORD_FREQ]
vocab_words = vocab_words[:MAX_VOCAB_SIZE]
id_to_token = SPECIAL_TOKENS + [word for word in vocab_words if word not in SPECIAL_TOKENS]
token_to_id = {token: idx for idx, token in enumerate(id_to_token)}

PAD_IDX = token_to_id[PAD_TOKEN]
UNK_IDX = token_to_id[UNK_TOKEN]
vocab_size = len(id_to_token)
sequence_len = min(max(len(tokens) - 1 for tokens in tokenized_lines), MAX_SEQUENCE_LEN_CAP)


def left_pad(ids: list[int], width: int, pad_idx: int = PAD_IDX) -> list[int]:
    ids = ids[-width:]
    return [pad_idx] * (width - len(ids)) + ids


def build_examples(lines: list[list[str]]) -> tuple[torch.Tensor, torch.Tensor]:
    features = []
    targets = []

    for tokens in lines:
        encoded = [token_to_id.get(token, UNK_IDX) for token in tokens]
        for end in range(2, len(tokens) + 1):
            target_token = tokens[end - 1]
            target_id = token_to_id.get(target_token)
            if target_id is None:
                continue

            prefix_ids = encoded[: end - 1]
            features.append(left_pad(prefix_ids, sequence_len))
            targets.append(target_id)

    if not features:
        raise ValueError('No training examples were created. Lower MIN_WORD_FREQ or add more corpus data.')

    return torch.tensor(features, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


X_train, y_train = build_examples(train_tokenized_lines)
X_val, y_val = build_examples(val_tokenized_lines)

print(f'Vocabulary size: {vocab_size}')
print(f'Model context length: {sequence_len}')
print(f'Train examples: {len(X_train)}')
print(f'Validation examples: {len(X_val)}')
print(f'Out-of-vocabulary train words: {sum(count for word, count in counter.items() if word not in token_to_id)}')


## Data Loaders


In [ ]:
BATCH_SIZE = 128

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=PIN_MEMORY,
)

print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')


## Model Architecture

The original model used a large dense projection tied to vocabulary size. Here the projection is fixed at 256 units, which keeps the classifier head much smaller as the corpus grows. `LayerNorm` is used instead of `BatchNorm` because it behaves more predictably for sequence models and small final batches.


In [ ]:
class ReemaBiLSTMPoemGenerator(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        sequence_len: int,
        embedding_dim: int = 128,
        hidden_dim: int = 128,
        final_hidden_dim: int = 256,
        dense_dim: int = 256,
        embedding_dropout: float = 0.10,
        dropout: float = 0.35,
        dropout_after_bi: float = 0.30,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(embedding_dropout)

        self.bi_lstm_1 = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.bi_lstm_2 = nn.LSTM(
            input_size=hidden_dim * 2,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.sequence_norm = nn.LayerNorm(hidden_dim * 2)
        self.dropout_after_bi = nn.Dropout(dropout_after_bi)

        self.final_lstm = nn.LSTM(
            input_size=hidden_dim * 2,
            hidden_size=final_hidden_dim,
            batch_first=True,
        )
        self.final_norm = nn.LayerNorm(final_hidden_dim)
        self.dropout_after_final = nn.Dropout(dropout)

        self.dense = nn.Linear(final_hidden_dim, dense_dim)
        self.dense_norm = nn.LayerNorm(dense_dim)
        self.dropout_after_dense = nn.Dropout(dropout)
        self.output = nn.Linear(dense_dim, vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        sequence, _ = self.bi_lstm_1(embedded)
        sequence, _ = self.bi_lstm_2(sequence)
        sequence = self.sequence_norm(sequence)
        sequence = self.dropout_after_bi(sequence)

        sequence, _ = self.final_lstm(sequence)
        pooled = sequence[:, -1, :]
        pooled = self.final_norm(pooled)
        pooled = self.dropout_after_final(pooled)

        features = F.relu(self.dense(pooled))
        features = self.dense_norm(features)
        features = self.dropout_after_dense(features)
        return self.output(features)


MODEL_CONFIG = {
    'sequence_len': sequence_len,
    'embedding_dim': 128,
    'hidden_dim': 128,
    'final_hidden_dim': 256,
    'dense_dim': 256,
    'embedding_dropout': 0.10,
    'dropout': 0.35,
    'dropout_after_bi': 0.30,
    'pad_idx': PAD_IDX,
}

model = ReemaBiLSTMPoemGenerator(vocab_size=vocab_size, **MODEL_CONFIG).to(device)


def count_parameters(model: nn.Module) -> tuple[int, int]:
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


total_params, trainable_params = count_parameters(model)
print(model)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')


## Training Helpers


In [ ]:
def perplexity(loss: float) -> float:
    return math.exp(min(float(loss), 20.0))


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch_inputs, batch_targets in loader:
        batch_inputs = batch_inputs.to(device, non_blocking=PIN_MEMORY)
        batch_targets = batch_targets.to(device, non_blocking=PIN_MEMORY)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(batch_inputs)
            loss = criterion(logits, batch_targets)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        batch_size = batch_targets.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == batch_targets).sum().item()
        total_count += batch_size

    avg_loss = total_loss / max(1, total_count)
    return {
        'loss': avg_loss,
        'accuracy': total_correct / max(1, total_count),
        'perplexity': perplexity(avg_loss),
    }


def safe_torch_load(path: Path, map_location: torch.device):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


## Train The Model

These settings are still small enough for Colab, but less prone to overfitting than the previous large dense head. If validation loss keeps improving at epoch 40, increase `EPOCHS`; if training loss drops while validation loss rises, increase dropout or add more data.


In [ ]:
EPOCHS = 40
LEARNING_RATE = 8e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 8
MIN_DELTA = 0.002
LABEL_SMOOTHING = 0.05

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

best_val_loss = math.inf
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, optimizer)
    val_metrics = run_epoch(model, val_loader, criterion)
    scheduler.step(val_metrics['loss'])

    record = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_accuracy': train_metrics['accuracy'],
        'train_perplexity': train_metrics['perplexity'],
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_perplexity': val_metrics['perplexity'],
        'learning_rate': optimizer.param_groups[0]['lr'],
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={record['train_loss']:.4f} | "
        f"train_acc={record['train_accuracy']:.4f} | "
        f"val_loss={record['val_loss']:.4f} | "
        f"val_acc={record['val_accuracy']:.4f} | "
        f"val_ppl={record['val_perplexity']:.2f}"
    )

    improved = val_metrics['loss'] < best_val_loss - MIN_DELTA
    if improved:
        best_val_loss = val_metrics['loss']
        epochs_without_improvement = 0
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'model_config': MODEL_CONFIG,
                'id_to_token': id_to_token,
                'token_to_id': token_to_id,
                'vocab_size': vocab_size,
            },
            WEIGHTS_PATH,
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping after {epoch} epochs.')
        break

checkpoint = safe_torch_load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Best validation loss: {best_val_loss:.4f}')
print(f'Best validation perplexity: {perplexity(best_val_loss):.2f}')


## Save Project Artifacts


In [ ]:
VOCAB_PATH.write_text(
    json.dumps(
        {
            'pad_token': PAD_TOKEN,
            'unk_token': UNK_TOKEN,
            'id_to_token': id_to_token,
            'token_to_id': token_to_id,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

CONFIG_PATH.write_text(
    json.dumps(
        {
            'model_config': MODEL_CONFIG,
            'preprocessing': {
                'min_word_freq': MIN_WORD_FREQ,
                'max_vocab_size': MAX_VOCAB_SIZE,
                'max_sequence_len_cap': MAX_SEQUENCE_LEN_CAP,
                'split_level': 'line',
            },
            'training': {
                'epochs_requested': EPOCHS,
                'epochs_completed': len(history),
                'batch_size': BATCH_SIZE,
                'learning_rate': LEARNING_RATE,
                'weight_decay': WEIGHT_DECAY,
                'validation_split': VALIDATION_SPLIT,
                'patience': PATIENCE,
                'min_delta': MIN_DELTA,
                'label_smoothing': LABEL_SMOOTHING,
                'best_val_loss': best_val_loss,
                'best_val_perplexity': perplexity(best_val_loss),
            },
            'data': {
                'source_path': str(DATA_PATH),
                'raw_lines': len(raw_lines),
                'usable_lines': len(tokenized_lines),
                'train_lines': len(train_tokenized_lines),
                'validation_lines': len(val_tokenized_lines),
                'vocab_size': vocab_size,
                'train_examples': len(X_train),
                'validation_examples': len(X_val),
                'sequence_len': sequence_len,
            },
            'source_repo': 'https://github.com/Reema-Khaseeb/NLP-poem-generator/tree/main',
            'framework': 'pytorch',
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding='utf-8')

print(f'Saved weights: {WEIGHTS_PATH}')
print(f'Saved vocabulary: {VOCAB_PATH}')
print(f'Saved config: {CONFIG_PATH}')
print(f'Saved history: {HISTORY_PATH}')


## Plot Training Curves


In [ ]:
if history:
    epochs = [record['epoch'] for record in history]

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [record['train_loss'] for record in history], label='train loss')
    plt.plot(epochs, [record['val_loss'] for record in history], label='validation loss')
    plt.xlabel('Epoch')
    plt.ylabel('Cross-entropy loss')
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [record['train_accuracy'] for record in history], label='train accuracy')
    plt.plot(epochs, [record['val_accuracy'] for record in history], label='validation accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Next-word accuracy')
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [record['train_perplexity'] for record in history], label='train perplexity')
    plt.plot(epochs, [record['val_perplexity'] for record in history], label='validation perplexity')
    plt.xlabel('Epoch')
    plt.ylabel('Perplexity')
    plt.legend()
    plt.show()


## Try Next-Word Generation

`predict_next_word` returns the most likely candidates. `generate_text` samples from the top-k candidates, which usually gives more varied poems than greedy argmax generation.


In [ ]:
def make_input_ids(text: str) -> torch.Tensor:
    tokens = tokenize(text)[-sequence_len:]
    ids = [token_to_id.get(token, UNK_IDX) for token in tokens]
    ids = left_pad(ids, sequence_len)
    return torch.tensor([ids], dtype=torch.long, device=device)


def next_word_logits(context: str) -> torch.Tensor:
    model.eval()
    with torch.no_grad():
        logits = model(make_input_ids(context))[0]
    logits = logits.clone()
    logits[PAD_IDX] = float('-inf')
    logits[UNK_IDX] = float('-inf')
    return logits


def predict_next_word(context: str, top_k: int = 5, temperature: float = 1.0) -> list[tuple[str, float]]:
    logits = next_word_logits(context)
    temperature = max(float(temperature), 1e-6)
    probs = F.softmax(logits / temperature, dim=-1)
    values, indices = torch.topk(probs, k=min(top_k, len(id_to_token)))
    return [(id_to_token[idx], float(prob)) for idx, prob in zip(indices.tolist(), values.tolist())]


def sample_next_word(context: str, top_k: int = 8, temperature: float = 0.9) -> str:
    logits = next_word_logits(context)
    top_k = min(max(1, int(top_k)), len(id_to_token))
    values, indices = torch.topk(logits, k=top_k)
    probs = F.softmax(values / max(float(temperature), 1e-6), dim=-1)
    sampled_position = torch.multinomial(probs, num_samples=1).item()
    return id_to_token[indices[sampled_position].item()]


def generate_text(seed_text: str, num_words: int = 25, top_k: int = 8, temperature: float = 0.9) -> str:
    text = seed_text.strip()
    for _ in range(num_words):
        text = f"{text} {sample_next_word(text, top_k=top_k, temperature=temperature)}"
    return text


seed_text = 'time was away'
print(predict_next_word(seed_text, top_k=5))
print(generate_text(seed_text, num_words=20, top_k=8, temperature=0.9))


## What Do The Weights Mean?

When the model summary says there are many weights, it means there are many learned scalar parameters. A weight is just a number the model adjusts during training to reduce next-word prediction error.

They are not one single human-readable characteristic like rhyme or tempo. Different groups of weights control different transformations:

- Embedding weights learn a vector representation for each word.
- LSTM weights learn how much previous context to remember, forget, and expose at each step.
- Dense/output weights map the final context representation to a probability score for every vocabulary word.
- LayerNorm weights learn small scaling/shifting corrections that stabilize activations.

So the characteristic being measured in the parameter count is capacity: the number of trainable numeric values. Training quality is measured separately with cross-entropy loss, perplexity, and next-word accuracy.
